In [2]:
import numpy as np
import pandas as pd

## Coding a spam classifier with naive Bayes

#### Loading the dataset

In [3]:
url = "https://raw.githubusercontent.com/luisguiserrano/manning/master/Chapter_08_Naive_Bayes/emails.csv"
emails = pd.read_csv(url)
emails.head()

,text,spam
0,Subject: naturally irresistible your corporate...,1
1,Subject: the stock trading gunslinger fanny i...,1
2,Subject: unbelievable new homes made easy im ...,1
3,Subject: 4 color printing special request add...,1
4,"Subject: do not have money , get software cds ...",1


_____
#### Pre-processing the email

We use the split() function to turn
the words into a list. We check only whether each word appears in
the email, regardless of how many times it appears, so we turn it
into a set and then into a list

In [4]:
def process_email(text):
    text = text.lower()
    return list(set(text.split())) 

emails["words"] = emails["text"].apply(process_email)

In [5]:
emails[:10]

,text,spam,words
0,Subject: naturally irresistible your corporate...,1,"[business, lt, image, guaranteed, promptness, ..."
1,Subject: the stock trading gunslinger fanny i...,1,"[attainder, tight, albeit, tanzania, continuan..."
2,Subject: unbelievable new homes made easy im ...,1,"[time, opportunity, we, being, website, uncond..."
3,Subject: 4 color printing special request add...,1,"[now, ), 5110, /, 4, form, format, ., phone, i..."
4,"Subject: do not have money , get software cds ...",1,"[do, ?, tradgedies, it, death, marriage, ., me..."
5,"Subject: great nnews hello , welcome to medzo...",1,"[we, 5, ieading, allusion, ., um, over, polite..."
6,Subject: here ' s a hot play in motion homela...,1,"[viewpoint, raw, ., what, paramount, person, ""..."
7,Subject: save your money buy getting this thin...,1,"[we, ?, now, thing, it, has, ship, ., buy, ove..."
8,Subject: undeliverable : home based business f...,1,"[business, 70590202251232, 32, ., 4, sent, 200..."
9,Subject: save your money buy getting this thin...,1,"[we, ?, now, thing, it, has, ship, ., buy, ove..."


____
#### Calculating the prior (probability that a random email is spam)

For this, we simply divide the number of spam emails by the total number of emails in the dataset.

In [6]:
num_emails = len(emails)
num_spam = sum(emails['spam'])

print("Number of emails:", num_emails)
print("Number of spam emails:", num_spam)
print()

# Calculating the prior probability that an email is spam
print("Probability of spam:", num_spam/num_emails)

Number of emails: 5728
Number of spam emails: 1368

Probability of spam: 0.2388268156424581


_____
#### Training a naive Bayes model

Our plan is to write a dictionary, and in this dictionary record every word, and its pair of occurrences in spam and ham. We also use Laplace smoothing, which prevents zero counts.

In [7]:
model = {}

for index, email in emails.iterrows():
    for word in email["words"]:
        if word not in model:
            model[word] = {"spam": 1, "ham": 1}
        if word in model:
            if email["spam"]:
                model[word]["spam"] += 1
            else:
                model[word]["ham"] += 1

In [8]:
model["lottery"]

{'spam': 9, 'ham': 1}

In [9]:
model["sale"]

{'spam': 39, 'ham': 42}

In [14]:
print(model)

{'business': {'spam': 325, 'ham': 650}, 'lt': {'spam': 28, 'ham': 1}, 'image': {'spam': 75, 'ham': 34}, 'guaranteed': {'spam': 96, 'ham': 15}, 'promptness': {'spam': 25, 'ham': 1}, 'make': {'spam': 276, 'ham': 711}, '.': {'spam': 1337, 'ham': 4323}, 'drafts': {'spam': 25, 'ham': 3}, 'clear': {'spam': 44, 'ham': 84}, 'become': {'spam': 83, 'ham': 112}, 'and': {'spam': 989, 'ham': 3569}, 'have': {'spam': 634, 'ham': 2800}, 'is': {'spam': 879, 'ham': 3060}, 'with': {'spam': 654, 'ham': 2663}, 'logo': {'spam': 97, 'ham': 13}, 'without': {'spam': 147, 'ham': 162}, 'are': {'spam': 618, 'ham': 2259}, 'change': {'spam': 100, 'ham': 213}, 'corporate': {'spam': 49, 'ham': 193}, 'extra': {'spam': 55, 'ham': 43}, 'more': {'spam': 399, 'ham': 883}, 'iogo': {'spam': 23, 'ham': 1}, 'all': {'spam': 472, 'ham': 1206}, 'list': {'spam': 243, 'ham': 400}, 'not': {'spam': 611, 'ham': 1568}, 'will': {'spam': 484, 'ham': 2459}, 'hand': {'spam': 82, 'ham': 68}, 'distinctive': {'spam': 26, 'ham': 2}, 'provide'

The counts start at 1 (to avoid zero counts), so *lottery* appears in 8 spam emails and 0 ham emails, and *sale* appears in 38 spam emails and 41 ham emails.

#### Using the model to make predictions

In [11]:
def predict_bayes(word):
    word = word.lower()
    num_spam_with_word = model[word]["spam"]
    num_ham_with_word = model[word]["ham"]
    return num_spam_with_word / (num_spam_with_word + num_ham_with_word)

In [12]:
predict_bayes("lottery")

0.9

In [13]:
predict_bayes("sale")

0.48148148148148145

#### Naive Bayes Prediction


In [28]:
def predict_naive_bayes(email):
    total = len(emails)
    num_spam = sum(emails["spam"])  # Count spam emails
    num_ham = total - num_spam      # Count ham emails

    email = email.lower()           # Convert to lowercase
    words = set(email.split())      # Extract unique words

    spams = [1.0]                   # Starting spam score
    hams = [1.0]                    # Starting ham score

    for word in words:
        if word in model:           # Only use words seen during training
            spams.append(model[word]["spam"] / num_spam)  # Word probability given spam
            hams.append(model[word]["ham"] / num_ham)     # Word probability given ham

    # Multiply word probabilities by class counts (class priors)
    prod_spams = float(np.prod(spams) * num_spam)
    prod_hams = float(np.prod(hams) * num_ham)

    # Normalize to get the estimated probability of spam
    return prod_spams / (prod_spams + prod_hams)

1. **Combine word probabilities:** Multiply the probabilities of the words in the email, assuming they are conditionally independent:

2. **Calculate class scores:** Multiply the word probabilities by their class priors:

   **Why?** The words tell us how strongly the email resembles each class, while the priors account for how common each class already is. Combining them gives a score that considers both the email's content and the overall class distribution.

5. **Normalize:** Divide the spam score by the sum of both scores:
   This converts the scores into an estimated probability between 0 and 1.

Notice that the model predicts high probability for the messages that sound spammy, and low probabilities for those that sound legitimate.

In [29]:
# Spammy
predict_naive_bayes('lottery sale')

0.9638144470140118

In [30]:
# Legit
predict_naive_bayes('Hi mom how are you')

0.1255435886716446

In [31]:
# Legit, but with unknown words
predict_naive_bayes('Hi MOM how aRe yoU afdjsaklfsdhgjasdhfjklsd')

0.1255435886716446

In [32]:
# Legit
predict_naive_bayes('meet me at the lobby of the hotel at nine am')

6.964603508395961e-05

In [33]:
# Spammy
predict_naive_bayes('enter the lottery to win three million dollars')

0.9995234218677428

In [34]:
#Spammy
predict_naive_bayes('buy cheap lottery easy money now')

0.999973472265966

In [35]:
# Legit (hopefully?) :)
predict_naive_bayes('Grokking Machine Learning by Luis Serrano')

0.41971076454845674

In [36]:
# Noise (which the model takes as an empty message).
predict_naive_bayes('asdfgh')

0.2388268156424581

In [37]:
# For the empty message, the probability is simply the prior
# (i.e., the number of spam emails divided by the total number of emails in the dataset)
predict_naive_bayes('')

0.2388268156424581